In [1]:
import matplotlib.pyplot as plt
import numpy as np
from obspy.imaging.beachball import aux_plane
import pandas as pd
from pathlib import Path

import sys
sys.path.append("../")
from src.synthetic_catalog_creation import synthetic_catalog_creation
from src.catalog_analysis import catalog_to_stress_and_instability
from src.instability_frequency import random_normals
from src.instability_frequency import instability_PDF
from src.plot import plot_unitmohr_with_coulomb
from src.plot import plot_instability_PDF

c:\Users\agata\miniconda3\envs\binst\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
output_dir = Path("../figures/instability_frequency")
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
### input parameters ###
sigma_model = np.array([[1,0,0], [0,-1,0], [0,0,0]])
Mmin = -2
Mc = 2
friction_model = 0.3
focmech_uncertainty = np.array([15, 34])    # degree

stressinverse_dir = "../../Stressinverse_1.1.3"

# relationship between instability and b-value
def model_i_to_b(instability, model_type):
    if model_type == "exp":
        return 1.1 - 0.4 * np.exp((instability - 1) / 0.4)
    if model_type == "linear":
        return np.where(
            instability >= 0.9,
            4 * (1 - instability) + 0.7,
            1.1
        )

In [4]:
columns = ["magnitude", "strike", "dip", "rake", "Snn", "Sne", "Snd", "See", "Sed", "Sdd", "R"]

catalog_Mmin, catalog_Mc = [
    pd.DataFrame(columns=columns)
    for _ in range(2)
]

In [5]:
### catalog creation (Mmin) ###
## distribution of the stress states on Mohr circle 
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "exp",
                                                Mmin,
                                                Mmin,
                                                friction_model,
                                                N_earthquake=10**4,
                                                N_subgroup=10**4
                                                )

normal_stress, shear_stress, _ = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_orig, dip_orig, rake_orig
)

fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction_model)

fig.savefig(output_dir/f"mohr_Mmin={Mmin}.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [6]:
## frequency distribution of synthetic earthquakes by instability (Mmin)
N = 10**6
n = random_normals(N, seed=1000)

fig, ax = plt.subplots(constrained_layout=True)
LEGEND_SIZE = 14
for _, R in enumerate(np.arange(0.1, 1, 0.4)):
    center, PDF = instability_PDF(R, friction_model, n)
    plot_instability_PDF(ax,  center, PDF, label=f"R={R:.1f}")
ax.set_ylim(-0.2, 8.2)
ax.legend(fontsize=LEGEND_SIZE)
fig.savefig(output_dir/f"instability_PDF_Mmin={Mmin}.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [7]:
### catalog creation (Mc) *same as catalog_no_error_exp ###
## distribution of stress states on Mohr circle
strike_orig, dip_orig, rake_orig, magnitude = synthetic_catalog_creation(
                                                sigma_model,
                                                model_i_to_b,
                                                "exp",
                                                Mmin,
                                                Mc,
                                                friction_model,
                                                N_earthquake=10**4,
                                                N_subgroup=10**6
                                                )

normal_stress, shear_stress, instability = catalog_to_stress_and_instability(
    sigma_model, friction_model, strike_orig, dip_orig, rake_orig
)

fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction_model)

fig.savefig(output_dir/f"mohr_Mc={Mc}.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [8]:
## frequency distribution of synthetic earthquakes by instability (Mc)
N = 10**6
n = random_normals(N, seed=1000)

fig, ax = plt.subplots(constrained_layout=True)
LEGEND_SIZE = 14
for _, R in enumerate(np.arange(0.1, 1, 0.4)):
    center, PDF = instability_PDF(R, friction_model, n)
    dI = center[1] - center[0]
    evolution = 10**(-model_i_to_b(center, "exp") * (Mc-Mmin))
    P = PDF * evolution / (np.sum(PDF * evolution) * dI)
    plot_instability_PDF(ax, center, P, label=f"R={R:.1f}")
ax.set_ylim(-0.2, 8.2)
ax.legend(fontsize=LEGEND_SIZE)
fig.savefig(output_dir/f"instability_PDF_Mc={Mc}.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [38]:
### Dataset1 ###
df = pd.read_csv("../data/california/processed/dataset1.csv")
sigma = df[
        ["Snn", "Sne", "Snd",
         "Sne", "See", "Sed",
         "Snd", "Sed", "Sdd"]
    ].to_numpy().reshape(-1, 3, 3)
friction = 0.32
strike1 = df["strike"].to_numpy()
dip1 = df["dip"].to_numpy()
rake1 = df["rake"].to_numpy()

n_events = len(strike1)
strike2, dip2, rake2 = np.zeros(n_events), np.zeros(n_events), np.zeros(n_events)
for k in range(n_events):
    strike2[k], dip2[k], rake2[k] = aux_plane(strike1[k], dip1[k], rake1[k])

normal_stress1, shear_stress1, instability1 = catalog_to_stress_and_instability(
    sigma, friction, strike1, dip1, rake1
)
normal_stress2, shear_stress2, instability2 = catalog_to_stress_and_instability(
    sigma, friction, strike2, dip2, rake2
)
mask = instability1 > instability2
normal_stress = np.where(mask, normal_stress1, normal_stress2)
shear_stress = np.where(mask, shear_stress1, shear_stress2)
instability = np.where(mask, instability1, instability2)

## distribution of stress states on Mohr circle
fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction)
fig.savefig(output_dir/f"mohr_dataset1.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability
fig, ax = plt.subplots(constrained_layout=True)
instability = np.array(instability)
density, bins = np.histogram(instability, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.2, 8.2)
fig.savefig(output_dir/f"instability_PDF_dataset1.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [ ]:
### Dataset2 ###
df = pd.read_csv("../data/california/processed/dataset2.csv")
sigma = df[
        ["Snn", "Sne", "Snd",
         "Sne", "See", "Sed",
         "Snd", "Sed", "Sdd"]
    ].to_numpy().reshape(-1, 3, 3)
friction = 0.35
strike1 = df["strike"].to_numpy()
dip1 = df["dip"].to_numpy()
rake1 = df["rake"].to_numpy()

n_events = len(strike1)
strike2, dip2, rake2 = np.zeros(n_events), np.zeros(n_events), np.zeros(n_events)
for k in range(n_events):
    strike2[k], dip2[k], rake2[k] = aux_plane(strike1[k], dip1[k], rake1[k])

normal_stress1, shear_stress1, instability1 = catalog_to_stress_and_instability(
    sigma, friction, strike1, dip1, rake1
)
normal_stress2, shear_stress2, instability2 = catalog_to_stress_and_instability(
    sigma, friction, strike2, dip2, rake2
)
mask = instability1 > instability2

normal_stress = np.where(mask, normal_stress1, normal_stress2)
shear_stress = np.where(mask, shear_stress1, shear_stress2)
instability = np.where(mask, instability1, instability2)

## distribution of stress states on Mohr circle
fig, ax = plot_unitmohr_with_coulomb(normal_stress, shear_stress, friction)
fig.savefig(output_dir/f"mohr_dataset2.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## frequency distribution of synthetic earthquakes by instability
fig, ax = plt.subplots(constrained_layout=True)
instability = np.array(instability)
density, bins = np.histogram(instability, bins=50, density=True)
centers = (bins[:-1]+bins[1:])/2
plot_instability_PDF(ax, centers, density)
ax.set_ylim(-0.2, 8.2)
fig.savefig(output_dir/f"instability_PDF_dataset2.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

In [31]:
print(f"strike1: {strike1[normal_stress+shear_stress>1]}")
print(f"dip1: {dip1[normal_stress+shear_stress>1]}")
print(f"rake1: {rake1[normal_stress+shear_stress>1]}")
print(f"normal stress: {normal_stress1[normal_stress+shear_stress>1]}")
print(f"shear stress: {shear_stress1[normal_stress+shear_stress>1]}")
print(f"instability: {instability1[normal_stress+shear_stress>1]}")

strike1: [226. 214. 330. 252. 221. 276.  26.  86. 225.  19. 240. 250.  38.  48.
 221. 267.  65. 254.  41. 206.  92. 233. 270.  33. 264. 231. 324.  56.
 172.  42. 230.  61. 234.  23.  22.  26. 221. 209. 215.  90.  68. 202.
 340. 267.  59.  56.  49.  34.  17.  57. 197.  19.   2. 185. 278. 350.
   0.  38.  21.  24.   0.  41.  19. 247.  63. 236. 254. 187. 218. 217.
  61. 246. 268. 270. 202.  71.  29.  51.  42.  35. 222.  40.  68.  25.
 249. 249.]
dip1: [70. 55. 10. 82. 43. 42. 74. 75. 64. 82. 53. 51. 54. 45. 63.  0. 66. 55.
 84. 50. 38. 68. 63. 54. 86. 49. 13. 41. 32. 69. 46. 75. 75. 51. 13. 46.
 84. 84. 73. 50. 77. 75.  6. 21. 51. 36. 72. 63. 39. 56. 69. 63. 72. 50.
  5. 33. 23. 85. 28. 35.  6. 76. 62. 40. 70. 43. 61.  4. 69. 47. 41. 81.
 37. 35. 59. 35. 24. 77. 79. 75. 55. 36. 54. 42. 22. 31.]
rake1: [  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0. -90.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0.   0.   0.   0.   0.   0.   0.   0.   

In [32]:
print(f"strike2: {strike2[normal_stress+shear_stress>1]}")
print(f"dip2: {dip2[normal_stress+shear_stress>1]}")
print(f"rake2: {rake2[normal_stress+shear_stress>1]}")
print(f"normal stress: {normal_stress2[normal_stress+shear_stress>1]}")
print(f"shear stress: {shear_stress2[normal_stress+shear_stress>1]}")
print(f"instability: {instability2[normal_stress+shear_stress>1]}")

strike2: [136. 124. 240. 162. 131. 186. 296. 356. 135. 289. 150. 160. 308. 318.
 131. 267. 335. 164. 311. 116.   2. 143. 180. 303. 174. 141. 234. 326.
  82. 312. 140. 331. 144. 293. 292. 296. 131. 119. 125.   0. 338. 112.
 250. 177. 329. 326. 319. 304. 287. 327. 107. 289. 272.  95. 188. 260.
 270. 308. 291. 294. 270. 311. 289. 157. 333. 146. 164.  97. 128. 127.
 331. 156. 178. 180. 112. 341. 299. 321. 312. 305. 132. 310. 338. 295.
 159. 159.]
dip2: [90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90.
 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90.
 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90.
 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90.
 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90. 90.]
rake2: [-160. -145. -100. -172. -133. -132. -164. -165. -154. -172. -143. -141.
 -144. -135. -153.  -90. -156. -145. -174. -140. -128. -158. -153. -144.
 -176. -139. -103. -131. -122. -159. -136. -

In [33]:
df[df["dip"]==0]

,year_event,month_event,day_event,hour_event,minute_event,second_event,latitude_event,longitude_event,depth_event,magnitude,...,rake,latitude_stress,longitude_stress,Snn,Sne,Snd,See,Sed,Sdd,R
22013,1992,4,28,9,59,50.896,33.93892,-116.34441,6.129,2.38,...,-90.0,33.925,-116.325,-0.529,-0.522,0.462,0.670,0.153,-0.141,0.608
33814,1993,10,5,2,47,33.105,32.43941,-115.05527,11.158,2.24,...,-90.0,32.425,-115.075,-0.905,-0.348,-0.109,0.288,0.239,0.617,0.681
35602,1994,1,18,12,35,30.610,34.30513,-118.66999,11.797,2.44,...,-90.0,34.325,-118.675,-0.896,-0.326,0.231,0.481,-0.083,0.416,0.828
69709,2019,7,6,6,58,11.184,35.90920,-117.74496,6.580,2.85,...,-90.0,35.925,-117.725,-0.931,-0.348,0.001,0.762,-0.026,0.169,0.638
69734,2019,7,6,7,23,43.842,35.87583,-117.68829,5.186,2.66,...,-90.0,35.875,-117.675,-0.612,-0.375,-0.391,0.755,-0.427,-0.143,0.534
71963,2019,7,23,8,50,37.814,35.59654,-117.47700,4.914,2.24,...,-90.0,35.575,-117.475,-0.739,-0.289,0.252,0.846,-0.295,-0.108,0.376
97100,1996,3,22,14,43,20.350,35.01033,-119.11450,16.220,2.45,...,-90.0,35.025,-119.125,-0.766,0.276,-0.467,0.420,0.124,0.346,0.965
112344,2005,7,24,3,1,12.270,38.80950,-122.81817,0.350,2.44,...,-90.0,38.825,-122.825,-0.178,-0.667,-0.085,0.616,-0.136,-0.439,0.184
122461,2015,3,27,7,33,12.560,36.22233,-120.34600,8.960,2.50,...,-90.0,36.225,-120.325,-0.706,-0.482,-0.341,-0.002,0.033,0.708,0.653
129140,2020,5,27,6,37,49.740,38.12033,-118.18317,3.150,2.90,...,-90.0,38.125,-118.175,-0.352,-0.680,0.386,0.265,0.280,0.088,0.760


In [36]:
# with pd.option_context("display.max_rows", None, "display.max_columns", None):
with pd.option_context("display.max_columns", None):
    display(df[normal_stress+shear_stress>1])

,year_event,month_event,day_event,hour_event,minute_event,second_event,latitude_event,longitude_event,depth_event,magnitude,strike,dip,rake,latitude_stress,longitude_stress,Snn,Sne,Snd,See,Sed,Sdd,R
1315,1982,4,27,15,42,40.388,35.76119,-117.73071,6.880,3.80,226.0,70.0,0.0,35.775,-117.725,-0.593,-0.235,-0.197,0.848,-0.409,-0.255,0.316
2994,1983,5,8,10,6,32.312,33.05682,-115.70784,9.197,3.01,214.0,55.0,0.0,33.075,-115.725,-0.811,0.290,0.324,0.946,-0.142,-0.135,0.496
3955,1983,12,28,23,23,8.203,35.94709,-118.31863,2.410,2.38,330.0,10.0,0.0,35.925,-118.325,0.107,0.292,-0.630,0.287,0.380,-0.395,0.943
5855,1984,12,31,15,10,50.314,33.52459,-116.79315,2.501,2.36,252.0,82.0,0.0,33.525,-116.775,-0.156,-0.696,-0.021,0.529,-0.279,-0.373,0.221
6868,1985,7,27,17,30,36.714,34.42137,-116.53920,6.946,2.47,221.0,43.0,0.0,34.425,-116.525,-0.150,-0.799,-0.386,-0.033,-0.104,0.183,0.689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127059,2019,7,12,13,34,58.770,36.06317,-117.85383,3.560,2.28,40.0,36.0,0.0,36.075,-117.875,-0.552,-0.214,-0.611,0.692,0.081,-0.140,0.689
127075,2019,7,13,18,14,35.290,36.18250,-117.90050,-0.860,3.03,68.0,54.0,0.0,36.175,-117.925,-0.786,-0.565,-0.158,0.730,-0.059,0.056,0.565
127347,2019,8,23,19,7,10.960,37.62467,-118.87417,7.850,2.29,25.0,42.0,0.0,37.625,-118.875,-0.464,0.307,-0.734,0.221,0.075,0.243,0.738
129086,2020,5,26,0,55,29.070,38.14367,-117.83500,5.260,2.90,249.0,22.0,0.0,38.125,-117.825,0.021,-0.919,-0.067,0.058,-0.355,-0.079,0.485
